### **Librerías**

In [7]:
#pip install GoogleNews
from GoogleNews import GoogleNews
import pandas as pd

https://pypi.org/project/GoogleNews/

### **Proceso**

In [2]:
# Configuramos GoogleNews con diferentes parámetros
googlenews = GoogleNews()
googlenews.set_lang('es')
googlenews.set_period('7d')
googlenews.set_encode('utf-8')

In [3]:
# Obtenemos las noticias de Google
googlenews.get_news('Gustavo Petro')
googlenews.results()

[]

## Método 1

In [9]:
# 1. Configuración
googlenews = GoogleNews(lang='es', encode='utf-8')

keywords = [
    "precio energía bolsa Colombia",
    "precio spot energía Colombia",
    "mercado de energía mayorista Colombia",
    "CREG energía Colombia",
    "escasez energía Colombia"
]

all_news = []

# 2. Extracción
for kw in keywords:
    googlenews.clear()
    googlenews.set_period('7d')  # luego lo automatizas por ventanas
    googlenews.get_news(kw)
    
    results = googlenews.results()
    
    for r in results:
        r['keyword'] = kw
        r['country'] = 'Colombia'
        r['domain'] = 'energia'
        all_news.append(r)

# 3. Construcción del corpus
corpus_df = pd.DataFrame(all_news)

# 4. Limpieza mínima
corpus_df.drop_duplicates(subset=['title', 'link'], inplace=True)

corpus_df.head()

""


In [ ]:
from GoogleNews import GoogleNews

googlenews = GoogleNews(lang='es', region='CO')
googlenews.enableException(True)

googlenews.set_time_range('01/01/2024', '01/31/2024')
googlenews.get_news('energía Colombia')

results = googlenews.results()

print("Noticias encontradas:", len(results))
results[:3]




Noticias encontradas: 0


[]

## Método 2

In [16]:
import pandas as pd
import requests
from io import BytesIO
#2

In [ ]:
#Palabras clave
keywords = [
    "energia",
    "precio energia",
    "bolsa de energia",
    "mercado electrico",
    "precio spot energia",
    "energia electrica"
]


In [19]:
# Establecer Rango de Fechas, Periodo
date_range = pd.period_range(start='2023-01', end='2025-12', freq='M')


In [20]:
def get_gdelt_mentions(keyword, start_date, end_date):
    url = (
        "https://api.gdeltproject.org/api/v2/doc/doc"
        f"?query={keyword}"
        f"&mode=ArtList"
        f"&maxrecords=250"
        f"&format=csv"
        f"&startdatetime={start_date}"
        f"&enddatetime={end_date}"
    )
    
    response = requests.get(url)
    
    if response.status_code != 200:
        return pd.DataFrame()
    
    df = pd.read_csv(BytesIO(response.content))
    return df


In [21]:
all_news = []

for period in date_range:
    start = period.to_timestamp('M').replace(day=1).strftime('%Y%m%d%H%M%S')
    end = period.to_timestamp('M').strftime('%Y%m%d%H%M%S')
    
    for kw in keywords:
        print(f"Descargando: {kw} | {start}–{end}")
        
        df = get_gdelt_mentions(kw, start, end)
        
        if df.empty:
            continue
        
        # Filtrar Colombia (campo Locations suele contener el país)
        df = df[df['Locations'].str.contains('Colombia', na=False)]
        
        if df.empty:
            continue
        
        df['keyword'] = kw
        df['month'] = period.strftime('%Y-%m')
        
        all_news.append(df)


Descargando: precio energía bolsa Colombia | 20230101000000–20230131000000
Descargando: precio spot energía Colombia | 20230101000000–20230131000000
Descargando: mercado de energía mayorista Colombia | 20230101000000–20230131000000
Descargando: CREG energía Colombia | 20230101000000–20230131000000
Descargando: escasez energía Colombia | 20230101000000–20230131000000
Descargando: precio energía bolsa Colombia | 20230201000000–20230228000000
Descargando: precio spot energía Colombia | 20230201000000–20230228000000
Descargando: mercado de energía mayorista Colombia | 20230201000000–20230228000000
Descargando: CREG energía Colombia | 20230201000000–20230228000000
Descargando: escasez energía Colombia | 20230201000000–20230228000000
Descargando: precio energía bolsa Colombia | 20230301000000–20230331000000
Descargando: precio spot energía Colombia | 20230301000000–20230331000000
Descargando: mercado de energía mayorista Colombia | 20230301000000–20230331000000
Descargando: CREG energía Colo

In [22]:
# Construcción del Corpus
corpus_df = pd.concat(all_news, ignore_index=True)

print("Corpus final:", corpus_df.shape)
corpus_df.head()


ValueError: No objects to concatenate

In [23]:
test_df = get_gdelt_mentions(
    keyword="energy",
    start_date="20240101000000",
    end_date="20240131000000"
)

print(test_df.shape)
test_df.head()


(0, 0)


""


## Método 3

In [24]:
import pandas as pd
import requests
import zipfile
from io import BytesIO

In [28]:
def load_gdelt_day(date):
    date_str = date.strftime('%Y%m%d')
    url = f"http://data.gdeltproject.org/events/{date_str}.export.CSV.zip"
    
    try:
        response = requests.get(url, timeout=30)
        z = zipfile.ZipFile(BytesIO(response.content))
        df = pd.read_csv(z.open(z.namelist()[0]), sep='\t', header=None)
        return df
    except Exception as e:
        print(f"Error en {date_str}: {e}")
        return pd.DataFrame()


In [29]:
test_date = pd.to_datetime("2024-01-15")
df_test = load_gdelt_day(test_date)

print(df_test.shape)
df_test.head()


(122137, 58)


,0,1,2,3,4,5,6,7,8,9,...,48,49,50,51,52,53,54,55,56,57
0,1151609375,20230115,202301,2023,2023.0411,BUS,CORPORATION,NaN,NaN,NaN,...,NaN,3,"Chesapeake Bay, Maryland, United States",US,USMD,38.5168,-76.3830,597227,20240115,https://smnewsnet.com/archives/528753/maryland...
1,1151609376,20230115,202301,2023,2023.0411,BUS,CORPORATION,NaN,NaN,NaN,...,NaN,2,"Maryland, United States",US,USMD,39.0724,-76.7902,MD,20240115,https://smnewsnet.com/archives/528753/maryland...
2,1151609377,20230115,202301,2023,2023.0411,BUS,CORPORATION,NaN,NaN,NaN,...,NaN,3,"Chesapeake Bay, Maryland, United States",US,USMD,38.5168,-76.3830,597227,20240115,https://smnewsnet.com/archives/528753/maryland...
3,1151609378,20230115,202301,2023,2023.0411,BUS,CORPORATION,NaN,NaN,NaN,...,NaN,2,"Maryland, United States",US,USMD,39.0724,-76.7902,MD,20240115,https://smnewsnet.com/archives/528753/maryland...
4,1151609379,20230115,202301,2023,2023.0411,KOR,SOUTH KOREA,KOR,NaN,NaN,...,-2960561,4,"Moscow, Moskva, Russia",RS,RS48,55.7522,37.6156,-2960561,20240115,https://www.crowrivermedia.com/national/news/n...


In [30]:
columns = [
    'GlobalEventID','Day','MonthYear','Year','FractionDate',
    'Actor1Code','Actor1Name','Actor1CountryCode',
    'Actor2Code','Actor2Name','Actor2CountryCode',
    'IsRootEvent','EventCode','EventBaseCode','EventRootCode',
    'QuadClass','GoldsteinScale','NumMentions','NumSources',
    'NumArticles','AvgTone',
    'Actor1Geo_Type','Actor1Geo_FullName','Actor1Geo_CountryCode',
    'Actor1Geo_ADM1Code','Actor1Geo_Lat','Actor1Geo_Long',
    'Actor1Geo_FeatureID',
    'Actor2Geo_Type','Actor2Geo_FullName','Actor2Geo_CountryCode',
    'Actor2Geo_ADM1Code','Actor2Geo_Lat','Actor2Geo_Long',
    'Actor2Geo_FeatureID',
    'ActionGeo_Type','ActionGeo_FullName','ActionGeo_CountryCode',
    'ActionGeo_ADM1Code','ActionGeo_Lat','ActionGeo_Long',
    'ActionGeo_FeatureID',
    'DATEADDED','SOURCEURL'
]

df_test.columns = columns
df_test.head()


ValueError: Length mismatch: Expected axis has 58 elements, new values have 44 elements

In [31]:
df_test.shape


(122137, 58)

In [32]:
len(df_test.columns)


58

In [33]:
df_test.head()


,0,1,2,3,4,5,6,7,8,9,...,48,49,50,51,52,53,54,55,56,57
0,1151609375,20230115,202301,2023,2023.0411,BUS,CORPORATION,NaN,NaN,NaN,...,NaN,3,"Chesapeake Bay, Maryland, United States",US,USMD,38.5168,-76.3830,597227,20240115,https://smnewsnet.com/archives/528753/maryland...
1,1151609376,20230115,202301,2023,2023.0411,BUS,CORPORATION,NaN,NaN,NaN,...,NaN,2,"Maryland, United States",US,USMD,39.0724,-76.7902,MD,20240115,https://smnewsnet.com/archives/528753/maryland...
2,1151609377,20230115,202301,2023,2023.0411,BUS,CORPORATION,NaN,NaN,NaN,...,NaN,3,"Chesapeake Bay, Maryland, United States",US,USMD,38.5168,-76.3830,597227,20240115,https://smnewsnet.com/archives/528753/maryland...
3,1151609378,20230115,202301,2023,2023.0411,BUS,CORPORATION,NaN,NaN,NaN,...,NaN,2,"Maryland, United States",US,USMD,39.0724,-76.7902,MD,20240115,https://smnewsnet.com/archives/528753/maryland...
4,1151609379,20230115,202301,2023,2023.0411,KOR,SOUTH KOREA,KOR,NaN,NaN,...,-2960561,4,"Moscow, Moskva, Russia",RS,RS48,55.7522,37.6156,-2960561,20240115,https://www.crowrivermedia.com/national/news/n...


In [34]:
columns = [
    'GlobalEventID','Day','MonthYear','Year','FractionDate',
    'Actor1Code','Actor1Name','Actor1CountryCode',
    'Actor2Code','Actor2Name','Actor2CountryCode',
    'IsRootEvent','EventCode','EventBaseCode','EventRootCode',
    'QuadClass','GoldsteinScale','NumMentions','NumSources',
    'NumArticles','AvgTone',
    'Actor1Geo_Type','Actor1Geo_FullName','Actor1Geo_CountryCode',
    'Actor1Geo_ADM1Code','Actor1Geo_Lat','Actor1Geo_Long',
    'Actor1Geo_FeatureID',
    'Actor2Geo_Type','Actor2Geo_FullName','Actor2Geo_CountryCode',
    'Actor2Geo_ADM1Code','Actor2Geo_Lat','Actor2Geo_Long',
    'Actor2Geo_FeatureID',
    'ActionGeo_Type','ActionGeo_FullName','ActionGeo_CountryCode',
    'ActionGeo_ADM1Code','ActionGeo_Lat','ActionGeo_Long',
    'ActionGeo_FeatureID',
    'DATEADDED','SOURCEURL'
]

len(columns)


44

In [37]:
df_test.shape[1]
df_test.columns = [f'col_{i}' for i in range(df_test.shape[1])]
df_test.head()


,col_0,col_1,col_2,col_3,col_4,col_5,col_6,col_7,col_8,col_9,...,col_48,col_49,col_50,col_51,col_52,col_53,col_54,col_55,col_56,col_57
0,1151609375,20230115,202301,2023,2023.0411,BUS,CORPORATION,NaN,NaN,NaN,...,NaN,3,"Chesapeake Bay, Maryland, United States",US,USMD,38.5168,-76.3830,597227,20240115,https://smnewsnet.com/archives/528753/maryland...
1,1151609376,20230115,202301,2023,2023.0411,BUS,CORPORATION,NaN,NaN,NaN,...,NaN,2,"Maryland, United States",US,USMD,39.0724,-76.7902,MD,20240115,https://smnewsnet.com/archives/528753/maryland...
2,1151609377,20230115,202301,2023,2023.0411,BUS,CORPORATION,NaN,NaN,NaN,...,NaN,3,"Chesapeake Bay, Maryland, United States",US,USMD,38.5168,-76.3830,597227,20240115,https://smnewsnet.com/archives/528753/maryland...
3,1151609378,20230115,202301,2023,2023.0411,BUS,CORPORATION,NaN,NaN,NaN,...,NaN,2,"Maryland, United States",US,USMD,39.0724,-76.7902,MD,20240115,https://smnewsnet.com/archives/528753/maryland...
4,1151609379,20230115,202301,2023,2023.0411,KOR,SOUTH KOREA,KOR,NaN,NaN,...,-2960561,4,"Moscow, Moskva, Russia",RS,RS48,55.7522,37.6156,-2960561,20240115,https://www.crowrivermedia.com/national/news/n...


In [38]:
gdelt = df_test[[
    'col_1',   # Day
    'col_14',  # EventRootCode
    'col_19',  # NumArticles
    'col_20',  # AvgTone
    'col_43',  # ActionGeo_CountryCode
    'col_57'   # SOURCEURL
]].copy()

gdelt.columns = [
    'Day',
    'EventRootCode',
    'NumArticles',
    'AvgTone',
    'CountryCode',
    'SourceURL'
]

gdelt.head()


,Day,EventRootCode,NumArticles,AvgTone,CountryCode,SourceURL
0,20230115,NaN,NaN,NaN,NaN,https://smnewsnet.com/archives/528753/maryland...
1,20230115,NaN,NaN,NaN,NaN,https://smnewsnet.com/archives/528753/maryland...
2,20230115,NaN,NaN,NaN,NaN,https://smnewsnet.com/archives/528753/maryland...
3,20230115,NaN,NaN,NaN,NaN,https://smnewsnet.com/archives/528753/maryland...
4,20230115,NaN,NaN,NaN,"Moscow, Moskva, Russia",https://www.crowrivermedia.com/national/news/n...


In [39]:
gdelt_co = gdelt[gdelt['CountryCode'] == 'CO']

print("Eventos Colombia:", gdelt_co.shape)
gdelt_co.head()


Eventos Colombia: (0, 6)


,Day,EventRootCode,NumArticles,AvgTone,CountryCode,SourceURL


In [40]:
gdelt_energy = gdelt_co[
    gdelt_co['EventRootCode'].isin(['01','02','03','04'])
]

print("Eventos energía/económicos:", gdelt_energy.shape)
gdelt_energy.head()


Eventos energía/económicos: (0, 6)


,Day,EventRootCode,NumArticles,AvgTone,CountryCode,SourceURL


## ESTE ES ##

In [22]:
#!pip install gnews pandas newspaper3k langdetect openpyxl
import pandas as pd
from gnews import GNews
from newspaper import Article
from langdetect import detect
from datetime import datetime
import time



In [30]:

#Configuración de la Conexión a Google News
google_news = GNews(
    language='es',
    country='CO',
    period='30d',          # Últimos 30 días
    max_results=100        # Ajustable
)
# Definición de Términos
queries = [
    "energía Colombia"
]
# "sector energético Colombia",
# "mercado eléctrico Colombia",
# "precio energía Colombia",
# "economía Colombia",
# "inflación Colombia",
# "crecimiento económico Colombia",
# "mercado financiero Colombia"


In [31]:
# Función para descargar el contenido completo del artículo
def descargar_articulo(url):
    try:
        article = Article(url, language='es')
        article.download()
        article.parse()
        return article.text.strip()
    except:
        return None


In [37]:
# Recolección y estructuración del dataset
data = []

for query in queries:
    print(f"\nBuscando noticias sobre: {query}")
    noticias = google_news.get_news(query)
    print("Noticias encontradas:", len(noticias))

    for noticia in noticias:
        texto = descargar_articulo(noticia["url"])
        if texto is None:
            continue

        registro = {
            "Fecha_Rec": datetime.now().strftime("%Y-%m-%d"),
            "Query": query,
            "Titulo": noticia.get("title"),
            "Fuente": noticia.get("publisher", {}).get("title"),
            "Fecha": noticia.get("published date"),
            "url": noticia.get("url"),
            "contenido": texto
        }

        data.append(registro)
        time.sleep(1)


Buscando noticias sobre: energía Colombia
Noticias encontradas: 100


In [39]:
df_raw = pd.DataFrame(data)

print("\nTotal de noticias recolectadas (RAW):", df_raw.shape[0])
df_raw.head()



Total de noticias recolectadas (RAW): 100


,Fecha_Rec,Query,Titulo,Fuente,Fecha,url,contenido
0,2026-01-22,energía Colombia,Colombia prioriza su soberanía energética y su...,Ministerio de Minas y Energía,"Thu, 22 Jan 2026 10:07:35 GMT",https://news.google.com/rss/articles/CBMi7wFBV...,
1,2026-01-22,energía Colombia,Colombia responde a Ecuador con la suspensión ...,EL PAÍS,"Thu, 22 Jan 2026 14:06:39 GMT",https://news.google.com/rss/articles/CBMi2wFBV...,
2,2026-01-22,energía Colombia,Colombia le apaga la luz a Ecuador: suspende v...,El Colombiano,"Thu, 22 Jan 2026 15:21:29 GMT",https://news.google.com/rss/articles/CBMiuwFBV...,
3,2026-01-22,energía Colombia,Gobierno cambiaría el esquema para la exportac...,Portafolio.co,"Tue, 13 Jan 2026 08:00:00 GMT",https://news.google.com/rss/articles/CBMixAFBV...,
4,2026-01-22,energía Colombia,Colombia suspende exportaciones de electricida...,Infobae,"Thu, 22 Jan 2026 12:56:15 GMT",https://news.google.com/rss/articles/CBMi8wFBV...,


In [50]:
# Convertir Fecha_Rec a datetime
df_raw["Fecha_Rec"] = pd.to_datetime(df_raw["Fecha_Rec"], errors="coerce")
# Crear Columnas
df_raw["Fecha"] = pd.to_datetime(
    df_raw["Fecha"],
    format="%a, %d %b %Y %H:%M:%S %Z",
    errors="coerce",
    utc=True
)
df_raw.rename(columns={"Fecha": "Fecha_Pub"}, inplace=True)
df_raw["Fecha"] = df_raw["Fecha_Pub"].dt.date
dias = {
    "Monday": "Lunes",
    "Tuesday": "Martes",
    "Wednesday": "Miércoles",
    "Thursday": "Jueves",
    "Friday": "Viernes",
    "Saturday": "Sábado",
    "Sunday": "Domingo"
}

df_raw["Dia"] = (
    df_raw["Fecha_Pub"]
    .dt.day_name()
    .map(dias)
)
df_raw["Hora"] = df_raw["Fecha_Pub"].dt.time



In [ ]:
# Eliminación de duplicados
df_clean = df_raw.drop_duplicates(subset="url").copy()
# Cálculo de longitud del texto
df_clean["longitud_texto"] = df_clean["contenido"].str.len()

# Dataset final listo para análisis
df_clean.reset_index(drop=True, inplace=True)

print("Total de noticias limpias:", df_clean.shape[0])
df_clean.head()



Total de noticias limpias: 100


,Fecha_Rec,Query,Titulo,Fuente,Fecha_Pub,url,contenido,Fecha,Dia,Hora,longitud_texto
0,2026-01-22,energía Colombia,Colombia prioriza su soberanía energética y su...,Ministerio de Minas y Energía,2026-01-22 10:07:35+00:00,https://news.google.com/rss/articles/CBMi7wFBV...,,2026-01-22,Jueves,10:07:35,0
1,2026-01-22,energía Colombia,Colombia responde a Ecuador con la suspensión ...,EL PAÍS,2026-01-22 14:06:39+00:00,https://news.google.com/rss/articles/CBMi2wFBV...,,2026-01-22,Jueves,14:06:39,0
2,2026-01-22,energía Colombia,Colombia le apaga la luz a Ecuador: suspende v...,El Colombiano,2026-01-22 15:21:29+00:00,https://news.google.com/rss/articles/CBMiuwFBV...,,2026-01-22,Jueves,15:21:29,0
3,2026-01-22,energía Colombia,Gobierno cambiaría el esquema para la exportac...,Portafolio.co,2026-01-13 08:00:00+00:00,https://news.google.com/rss/articles/CBMixAFBV...,,2026-01-13,Martes,08:00:00,0
4,2026-01-22,energía Colombia,Colombia suspende exportaciones de electricida...,Infobae,2026-01-22 12:56:15+00:00,https://news.google.com/rss/articles/CBMi8wFBV...,,2026-01-22,Jueves,12:56:15,0


In [54]:
from newspaper import Article, Config

config = Config()
config.browser_user_agent = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)
config.request_timeout = 10

article = Article(url, language="es", config=config)
article.download()
article.parse()

texto = article.text
print("Texto:", len(texto))


Texto: 0


In [56]:
def obtener_contenido(url):
    try:
        article = Article(url, language="es", config=config)
        article.download()
        article.parse()

        texto = article.text.strip()

        # Si newspaper3k no extrae nada, usar fallback
        if texto == "":
            headers = {
                "User-Agent": (
                    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                    "AppleWebKit/537.36 (KHTML, like Gecko) "
                    "Chrome/120.0.0.0 Safari/537.36"
                )
            }
            r = requests.get(url, headers=headers, timeout=10)
            soup = BeautifulSoup(r.text, "html.parser")

            parrafos = soup.find_all("p")
            texto = " ".join(p.get_text(strip=True) for p in parrafos)

        return texto.strip()

    except Exception as e:
        return ""



In [58]:
df_raw["Contenido"] = df_raw["url"].apply(obtener_contenido)


In [61]:
df_raw[["url", "Contenido"]].head()
df_raw["Contenido"].str.len().describe()



count    100.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
Name: Contenido, dtype: float64

In [62]:
import requests

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

r = requests.get(df_raw["url"].iloc[0], headers=headers)
print(r.text[:2000])


<!doctype html><html lang="en-US" dir="ltr"><head><base href="https://news.google.com/"><link rel="preconnect" href="//www.gstatic.com"><meta name="referrer" content="origin"><link rel="canonical" href="https://news.google.com/rss/articles/CBMi7wFBVV95cUxOUnJ4N0pZelpvRzJWY2k5VmNKVE5uRXpUdUVxa1dHOG1mUTBhRkhaVnRNWGNpd3JHX3Q2MFZhVkRqT1RvTlJObzhkajQ0U2daaERHLXVnT1U1QlhUY1owRHMzZFoyWmI5R0owMzFPcFBWXzBfelZLTlpHdXRtZld1em9jMzNhbE1YYkNxTGo2LVp6aEVZclluTXFhMnVXZmtKb21OWml1bURwWUZHbjAtNVBwZHRmWWNXV0RWRGJ4MkJMd2dlV0VtNU5jdzQ0WmMxdTBTMVdCNGNCa004VVNoajBiTkpEZENTVUhQc3lYQQ"><meta name="viewport" content="width=device-width,initial-scale=1,minimal-ui,viewport-fit=cover"><meta name="google-site-verification" content="AcBy5YFny2HQgVUCR18tO5YUTf6MpVlcJqGTd-a9-SI"><meta name="mobile-web-app-capable" content="yes"><meta name="apple-mobile-web-app-capable" content="yes"><meta name="application-name" content="News"><meta name="apple-mobile-web-app-title" content="News"><meta name="apple-mobile-web-app-stat

In [63]:
df_raw["url"].iloc[0]

'https://news.google.com/rss/articles/CBMi7wFBVV95cUxOUnJ4N0pZelpvRzJWY2k5VmNKVE5uRXpUdUVxa1dHOG1mUTBhRkhaVnRNWGNpd3JHX3Q2MFZhVkRqT1RvTlJObzhkajQ0U2daaERHLXVnT1U1QlhUY1owRHMzZFoyWmI5R0owMzFPcFBWXzBfelZLTlpHdXRtZld1em9jMzNhbE1YYkNxTGo2LVp6aEVZclluTXFhMnVXZmtKb21OWml1bURwWUZHbjAtNVBwZHRmWWNXV0RWRGJ4MkJMd2dlV0VtNU5jdzQ0WmMxdTBTMVdCNGNCa004VVNoajBiTkpEZENTVUhQc3lYQQ?oc=5&hl=en-US&gl=US&ceid=US:en'